# WAF Assessment — Configuration

**Run this first, or `%run` it from a pillar notebook.** Every pillar notebook and the
driver notebook load their settings from here, so this is the single place to configure
the assessment.

| Setting | What it does |
| --- | --- |
| `CATALOG` / `SCHEMA` | Where results are persisted so pillars can be combined |
| `PROD_CATALOG_PATTERNS` | Which catalogs count as production |
| `LOOKBACK_DAYS` | Activity window for telemetry-based checks |
| `COMPLETE_AT` | Coverage needed to score `completed` (default 80%) |

Fill in the **REQUIRED** values below before running anything.

## 1. Required settings

`CATALOG` and `SCHEMA` are where the assessment writes its own results table. Pick a
location you can write to — the assessment never writes anywhere else.

In [ ]:
# ============================================================================
# REQUIRED - fill these in
# ============================================================================

# Catalog and schema for the assessment's own results table.
# The schema is created if missing; the catalog must already exist unless you
# set CREATE_CATALOG = True below.
CATALOG = ""   # e.g. "main"
SCHEMA = ""    # e.g. "waf_assessment"

# ============================================================================
# Production scope - review these
# ============================================================================

# Catalogs matching any of these SQL LIKE patterns are treated as production.
# Governance questions are scored against production because that is where it
# matters. If NOTHING matches, the notebooks fall back to all non-system
# catalogs and stamp a caveat on every affected rationale.
PROD_CATALOG_PATTERNS = ["prod%", "main", "%_prod", "%_production"]

# Schemas matching these patterns are excluded even inside a production catalog.
PROD_SCHEMA_EXCLUDE_PATTERNS = ["%dev%", "%test%", "%sandbox%", "%tmp%", "%scratch%"]

# Never assessed as customer data.
SYSTEM_CATALOGS = ["system", "__databricks_internal%", "samples"]

# ============================================================================
# Scoring - defaults follow the WAF Assessment Tool convention
# ============================================================================

# Activity window for telemetry checks (job runs, queries, billing, lineage).
LOOKBACK_DAYS = 90

# Coverage at or above this fraction scores "completed"; below scores
# "in-progress". 0.80 = the 80% rule.
COMPLETE_AT = 0.80

# ============================================================================
# Optional
# ============================================================================

CREATE_CATALOG = False       # True to CREATE CATALOG IF NOT EXISTS
RESULTS_TABLE = "waf_assessment_results"
PERSIST_RESULTS = True       # False to run read-only with no results table
DEBUG = False                # True to print tracebacks from failing checks
RUN_ID = None                # None generates a timestamped id

## 2. Validation and shared context

This cell validates your settings, makes the repo importable, and builds the shared
`ctx` object the checks run against. It fails loudly if a required value is missing.

In [ ]:
import os
import sys

# Make the repo root importable regardless of where this notebook lives.
_here = os.path.dirname(os.path.abspath("__file__" if "__file__" not in dir() else __file__))
try:
    _nb_dir = os.path.dirname(
        dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        .notebookPath().get()
    )
    _candidates = ["/Workspace" + _nb_dir, "/Workspace" + os.path.dirname(_nb_dir), _here]
except Exception:
    _candidates = [_here, os.getcwd()]

REPO_ROOT = None
for _c in _candidates:
    if _c and os.path.exists(os.path.join(_c, "waf_core.py")):
        REPO_ROOT = _c
        break
    _parent = os.path.dirname(_c) if _c else None
    if _parent and os.path.exists(os.path.join(_parent, "waf_core.py")):
        REPO_ROOT = _parent
        break

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate waf_core.py. Ensure this notebook is inside the "
        "waf_current_state_scripts repo (Git folder) and that waf_core.py sits at "
        "the repo root."
    )
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# --- validate ---------------------------------------------------------------
_missing = [n for n, v in [("CATALOG", CATALOG), ("SCHEMA", SCHEMA)] if not str(v).strip()]
if _missing and PERSIST_RESULTS:
    raise ValueError(
        f"Set {' and '.join(_missing)} in the cell above before running. "
        "(Or set PERSIST_RESULTS = False to run without a results table.)"
    )
if not 0 < COMPLETE_AT <= 1:
    raise ValueError(f"COMPLETE_AT must be between 0 and 1, got {COMPLETE_AT}")
if int(LOOKBACK_DAYS) < 1:
    raise ValueError(f"LOOKBACK_DAYS must be at least 1, got {LOOKBACK_DAYS}")

import waf_core as wc
import waf_questions as wq

CFG = {
    "catalog": CATALOG.strip(),
    "schema": SCHEMA.strip(),
    "results_table": RESULTS_TABLE,
    "prod_catalog_patterns": PROD_CATALOG_PATTERNS,
    "prod_schema_exclude_patterns": PROD_SCHEMA_EXCLUDE_PATTERNS,
    "system_catalogs": SYSTEM_CATALOGS,
    "lookback_days": int(LOOKBACK_DAYS),
    "complete_at": float(COMPLETE_AT),
    "create_catalog": bool(CREATE_CATALOG),
    "persist_results": bool(PERSIST_RESULTS),
    "debug": bool(DEBUG),
    "run_id": RUN_ID,
}

ctx = wc.Ctx(CFG, spark=spark)

print(f"Repo root      : {REPO_ROOT}")
print(f"Run ID         : {ctx.run_id}")
print(f"Lookback       : {ctx.lookback_days} days")
print(f"Complete at    : {ctx.complete_at:.0%}")
if PERSIST_RESULTS:
    print(f"Results table  : {wc.results_table_fqn(CFG)}")
else:
    print("Results table  : (disabled)")

## 3. Resolve assessment scope

Resolving the scope now surfaces a mis-set `PROD_CATALOG_PATTERNS` immediately, rather
than as a puzzling wall of `open` results later.

In [ ]:
scope = ctx.scope

print(f"Scope mode : {scope.mode}")
print(f"Catalogs   : {len(scope.catalogs)}")
for c in scope.catalogs[:25]:
    print(f"   - {c}")
if len(scope.catalogs) > 25:
    print(f"   ... and {len(scope.catalogs) - 25} more")

if scope.mode == "all":
    print(
        "\nWARNING: no catalog matched PROD_CATALOG_PATTERNS "
        f"({PROD_CATALOG_PATTERNS}).\n"
        "Falling back to ALL non-system catalogs. Every affected rationale will say so.\n"
        "If these results should reflect production only, update "
        "PROD_CATALOG_PATTERNS above and re-run."
    )
elif not scope.catalogs:
    print("\nWARNING: scope is empty - no catalogs to assess.")
else:
    print("\nScope resolved to production catalogs.")

## 4. Verify system table access

Checks degrade to `open` with an explanatory rationale when a system schema is
unreadable, so the assessment still runs — but a lot of `open` results usually means
missing grants rather than missing controls. Confirm here before drawing conclusions.

In [ ]:
REQUIRED_TABLES = [
    ("system.information_schema.tables", "catalog metadata (most pillars)"),
    ("system.information_schema.columns", "column metadata / documentation"),
    ("system.access.audit", "audit logging (governance, ops)"),
    ("system.access.table_lineage", "lineage (governance, interoperability)"),
    ("system.billing.usage", "cost and serverless adoption"),
    ("system.compute.clusters", "compute configuration"),
    ("system.compute.warehouses", "SQL warehouse configuration"),
    ("system.query.history", "query performance and monitoring"),
    ("system.lakeflow.jobs", "job orchestration and reliability"),
    ("system.lakeflow.pipelines", "declarative pipelines"),
    ("system.storage.table_metrics_history", "file sizes, predictive optimization"),
    ("system.mlflow.runs_latest", "ML experiment tracking"),
    ("system.serving.served_entities", "model serving"),
]

available, unavailable = [], []
for fqn, why in REQUIRED_TABLES:
    (available if ctx.has_table(fqn) else unavailable).append((fqn, why))

print(f"Readable: {len(available)}/{len(REQUIRED_TABLES)} system tables\n")
for fqn, why in available:
    print(f"  [x] {fqn:<50} {why}")
if unavailable:
    print()
    for fqn, why in unavailable:
        print(f"  [ ] {fqn:<50} {why}")
    print(
        "\nUnreadable tables produce 'open' results with an explanatory rationale.\n"
        "If that is unexpected, grant SELECT on the system schemas to the principal "
        "running this assessment."
    )

## 5. Helper used by the pillar notebooks

`run_pillar` is the one entry point each pillar notebook calls. Keeping it here means the
pillar notebooks stay thin and consistent.

In [ ]:
import importlib


def run_pillar(module_name: str, pillar_id: str, persist: bool = None):
    """Run one pillar's checks, print its report, optionally persist, return results.

    Args:
        module_name: module under ``checks/`` (e.g. "checks.governance").
        pillar_id: WAF pillar id (e.g. "data-ai-governance").
        persist: override ``PERSIST_RESULTS`` for this run.
    """
    mod = importlib.import_module(module_name)
    importlib.reload(mod)  # pick up edits without detaching the notebook

    pillar_name = wq.PILLAR_NAMES[pillar_id]
    meta = wq.pillar_meta(pillar_id)

    print(f"Assessing {pillar_name}: {len(mod.CHECKS)} question(s)\n")
    results = ctx.run_checks(pillar_name, meta, mod.CHECKS)

    # Fail loudly if a pillar's CHECKS list drifts from the published question bank.
    expected, got = set(meta), {r.qid for r in results}
    if expected - got:
        raise AssertionError(f"{pillar_name}: no check defined for {sorted(expected - got)}")
    if got - expected:
        raise AssertionError(f"{pillar_name}: unknown question ids {sorted(got - expected)}")

    print()
    print(wc.render_pillar_report(pillar_name, results))

    should_persist = CFG["persist_results"] if persist is None else persist
    if should_persist:
        wc.persist_results(ctx, results)
    return results


print("Configuration loaded. run_pillar() is ready.")